# Contextual Compression Retriever

- Author: [JoonHo Kim](https://github.com/jhboyo)
- Design: []()
- Peer Review :
- This is a part of [LangChain Open Tutorial](https://github.com/LangChain-OpenTutorial/LangChain-OpenTutorial)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LangChain-OpenTutorial/LangChain-OpenTutorial/blob/main/06-DocumentLoader/04-CSV-Loader.ipynb) [![Open in GitHub](https://img.shields.io/badge/Open%20in%20GitHub-181717?style=flat-square&logo=github&logoColor=white)](https://github.com/LangChain-OpenTutorial/LangChain-OpenTutorial/blob/main/06-DocumentLoader/04-CSV-Loader.ipynb)


## Overview

The `ContextualCompressionRetriever` in LangChain is a powerful tool designed to optimize the retrieval process by compressing retrieved documents based on context. This retriever is particularly useful in scenarios where large amounts of data need to be summarized or filtered dynamically, ensuring that only the most relevant information is passed to subsequent processing steps.

Key features of the ContextualCompressionRetriever include:

- Context-Aware Compression: Documents are compressed based on the specific context or query, ensuring relevance and reducing redundancy.
- Flexible Integration: Works seamlessly with other LangChain components, making it easy to integrate into existing pipelines.
- Customizable Compression: Allows for the use of different compression techniques, including summary models and embedding-based methods, to tailor the retrieval process to your needs.

The `ContextualCompressionRetriever` is particularly suited for applications like:

- Summarizing large datasets for Q&A systems.
- Enhancing chatbot performance by providing concise and relevant responses.
- Improving efficiency in document-heavy tasks like legal analysis or academic research.

By using this retriever, developers can significantly reduce computational overhead and improve the quality of information presented to end-users.

![](./assets/02-contextual-compression-retriever-workflow.png)  


### Table of Contents

- [Overview](#overview)
- [Environment Setup](#environment-setup)
- [Basic Retriever Configuration](#basic-retriever-configuration)
- [Contextual Compression](#contextual-compression)
- [Document Filtering Using LLM](#document-filtering-using-llm)
- [Creating a Pipeline (Compressor + Document Converter)](#creating-a-pipeline-compressor--document-converter)


### References

- [How to do retrieval with contextual compression](https://python.langchain.com/docs/how_to/contextual_compression/)
- [LLM ChainFilter](https://python.langchain.com/api_reference/langchain/retrievers/langchain.retrievers.document_compressors.chain_filter.LLMChainFilter.html)

----

In [1]:
%%capture --no-stderr
%pip install langchain-opentutorial

In [2]:
# Install required packages
from langchain_opentutorial import package

package.install(
    [
        "langchain",
        "langchain_openai",
        "langchain_community",
        "langchain_text_splitters",
        "langchain_core",
        "faiss-cpu",
    ],
    verbose=False,
    upgrade=False,
)


[notice] A new release of pip is available: 24.0 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [31]:
# Set environment variables
from langchain_opentutorial import set_env

set_env(
    {
        "OPENAI_API_KEY": "",
        "LANGCHAIN_API_KEY": "",
        "LANGCHAIN_TRACING_V2": "true",
        "LANGCHAIN_ENDPOINT": "https://api.smith.langchain.com",
        "LANGCHAIN_PROJECT": "Contextual Compression Retriever",
    }
)

Environment variables have been set successfully.


You can alternatively set `OPENAI_API_KEY` in `.env` file and load it. 

[Note] This is not necessary if you've already set `OPENAI_API_KEY` in previous steps.

In [12]:
from dotenv import load_dotenv

# Load API KEY information
load_dotenv(override=True)

True

The following function is used to display documents in a visually appealing format.


In [13]:
# Helper function to print documents in a pretty format
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

## Basic Retriever Configuration

Let's start by initializing a simple vector store retriever and saving text documents in chunks.
When a sample question is asked, you can see that the retriever returns 1 to 2 relevant documents along with a few irrelevant ones.

We will follow the following steps to generate a retriever.
1. TextLoader를 사용하여 텍스트 파일을 로드하는 로더 생성
2. CharacterTextSplitter를 사용하여 텍스트 청크를 생성하고 텍스트를 겹치지 않는 100 청크로 분할합니다.
3. FAISS를 사용하여 벡터 저장소를 생성하고 이를 검색기로 변환합니다.
4. 검색기에 쿼리하여 관련 문서를 찾습니다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter

# 1. Generate Loader to lthe text file using TextLoader
loader = TextLoader("./data/test.txt")\

# 2. Generate text chunks using CharacterTextSplitter and split the text into chunks of 300 characters with no overlap.
text_splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = loader.load_and_split(text_splitter)

# 3. Generate vector store using FAISS and convert it to retriever
retriever = FAISS.from_documents(texts, OpenAIEmbeddings()).as_retriever()

# 4. Query the retriever to find relevant documents
docs = retriever.invoke("애국가 알려줘?")

# 5. Print the relevant documents
pretty_print_docs(docs)


'''
[ 출력 결과 ]
document 1:

애국가
애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.
오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.
그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.
한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.
----------------------------------------------------------------------------------------------------
document 2:

애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 3:

애국가 4절
이 기상과 이 맘으로 충성을 다하여
괴로우나 즐거우나 나라 사랑하세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 4:

우리나라 이름 
우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요.
영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.
Failed to multipart ingest runs: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=a4a2a678-192a-4660-890e-18f4c5cd544f,id=a4a2a678-192a-4660-890e-18f4c5cd544f

'''

## Contextual Compression 

검색 시스템에서 직면하는 어려움 중 하나는 데이터를 시스템에 수집할 때 어떤 특정 질의를 처리해야 할지 미리 알 수 없다는 점입니다.

→ 사용자가 어떤 데이터 와 키워드를 가지고 검색 할지 미리 파악이 어렵기 때문이다. 

이러한 전체 문서를 애플리케이션에 전달하면 더 비용이 많이 드는 LLM 호출과 품질이 낮은 응답으로 이어질 수 있습니다.

`ContextualCompressionRetriever` 은 이 문제를 해결하기 위해 만들어졌다. 

`ContextualCompressionRetriever` 은 검색된 문서를 그대로 즉시 반환하는대신 주어진 질의 맥락을 사용하여 문서를 압축함으로써 관련된 정보만 반환되도록 하며 
여기서 말하는 “압축” 은 개별 문서의 내용 압축과 문서를 전체적으로 필터하는것을 의미한다.


In [19]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI

# 1. Open AI 언어 모델 초기화
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")  

# 2. LLM을 사용하여 문서 압축기 생성
compressor = LLMChainExtractor.from_llm(llm)

# 3. ContextualCompressionRetriever를 통해 압축 검색기 생성 
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

# 4. 압축 검색기로 질의
compressed_docs = (
    compression_retriever.invoke( 
        "안녕 나는 미국에서 대한민국으로 이민을 왔는데 국가를 모르겠어 애국가를 알려줘"
    )
)
print("=========================================================")
print("============== LLMChainExtractor 적용 후 ==================")

# 5. Print the relevant documents
pretty_print_docs(compressed_docs)

'''
[ 출력 결과 ]

=========================================================
============== LLMChainExtractor 적용 후 ==================
document 1:

애국가 4절  
이 기상과 이 맘으로 충성을 다하여  
괴로우나 즐거우나 나라 사랑하세  
무궁화 삼천리 화려 강산  
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 2:

애국가
애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.
오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.
그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.
한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.
----------------------------------------------------------------------------------------------------
document 3:

애국가 1절  
동해물과 백두산이 마르고 닳도록  
하느님이 보우하사 우리나라 만세  
무궁화 삼천리 화려 강산  
대한 사람 대한으로 길이 보전하세
'''

============== LLMChainExtractor 적용 후 ==================
document 1:

애국가 4절  
이 기상과 이 맘으로 충성을 다하여  
괴로우나 즐거우나 나라 사랑하세  
무궁화 삼천리 화려 강산  
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 2:

애국가
애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.
오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.
그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.
한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.
----------------------------------------------------------------------------------------------------
document 3:

애국가 1절  
동해물과 백두산이 마르고 닳도록  
하느님이 보우하사 우리나라 만세  
무궁화 삼천리 화려 강산  
대한 사람 대한으로

## Document Filtering Using LLM


### LLMChainFilter

LLMChainFilter 는 초기에 검색된 문서중 어떤 문서를 필터링하고 어떤 문서를 반환할지 결정하기 위해 LLM 체인을 사용합니다.

In [20]:
from langchain.retrievers.document_compressors import LLMChainFilter

# 1. LLM Chain을 이용해 LLMChainFilter 객체 생성
_filter = LLMChainFilter.from_llm(llm)

# 2. ContextualCompressionRetriever를 통해 압축 검색기 생성 
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter,
    base_retriever=retriever,
)

# 3. 문서 검색
compressed_docs = compression_retriever.invoke(
    "안녕 나는 미국에서 대한민국으로 이민을 왔는데 국가를 모르겠어 애국가를 알려줘"
)

# 4. Print the relevant documents
pretty_print_docs(compressed_docs)  

document 1:

애국가 4절
이 기상과 이 맘으로 충성을 다하여
괴로우나 즐거우나 나라 사랑하세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 2:

애국가
애국가(愛國歌)는 ‘나라를 사랑하는 노래’라는 뜻이에요. 우리나라는 애국가에 특별한 이름을 붙이지 않고 국가(國歌)로 사용하고 있어요.
오늘날 불리고 있는 애국가 노랫말은 우리나라가 외세의 침략으로 위기에 처해있던 시기(1907년 전후)에 나라 사랑하는 마음과 우리 민족의 자주의식을 북돋우기 위해 만들어진 것으로 보여져요.
그 후 여러 선각자의 손을 거쳐 현재와 같은 내용을 담게 되었는데 이 노랫말에 붙여진 곡조는 스코틀랜드 민요 ‘올드 랭 사인 (Auld Lang Syne)’ 이었답니다. 당시 해외에서 활동 중이던 작곡가 안익태(安益泰) 선생은 애국가에 남의 나라 곡을 붙여 부르는 것을 안타깝게 여겨 1935년에 오늘날의 애국가를 작곡하였다고 해요.
1948년 대한민국 정부가 수립된 이후 현재의 애국가가 정부의 공식행사에서 불려지고, 교과서에도 실리면서 전국적으로 불려지기 시작했답니다.
한 세기 가까운 세월 동안 슬플 때나 기쁠 때나 우리 겨레와 운명을 같이 해 온 애국가를 부를 때마다 우리 모두 선조들의 나라 사랑 정신을 새롭게 되새겨보아요.
----------------------------------------------------------------------------------------------------
document 3:

애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세


### EmbeddingsFilter

문서와 쿼리를 임베딩하고 쿼리와 충분히 유사한 임베딩을 가진 문서만 반환함으로 더 저렴하고 빠른 옵션 제공 
→ 임베딩 필터를 이용하면 계산 비용과 시간 절약이 가능하다. 

In [28]:
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import OpenAIEmbeddings

# 1. 임베딩 모델 로드 
embeddings = OpenAIEmbeddings()

# 2. 유사도 임계값이 0.80인 EmbeddingsFilter 객체를 생성합니다
embeddings_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.80)

# 3. 기본 압축기로 embeddings_filter를, 기본 검색기로 retriever를 사용하여 ContextualCompressionRetriever 객체를 생성합니다.
compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter, 
    base_retriever=retriever
)

# 4. 검색 질의
compressed_docs = compression_retriever.invoke(
    "우리나라의 정식 이름의 대해 알려줘"
)

# 5. Print the relevant documents
pretty_print_docs(compressed_docs)

document 1:

우리나라 이름 
우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요.
영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.
----------------------------------------------------------------------------------------------------
document 2:

대한민국 나라문장
문장은 외국에서 특정 가문이나 단체 또는 국가의 권위를 상징하는 장식적인 표시로 한 나라의 문장은 ‘국가 문장’ 또는 ‘국장’이라고 해요.
우리나라의 문장은 태극문양을 무궁화 꽃잎 5장이 감싸고 ‘대한민국’ 글자가 새겨진 리본으로 그 테두리를 둘러싸고 있어요.
1963년 12월 10일 ‘나라문장규정’을 제정하고, 외국기관에 발송되는 중요문서, 훈장 및 대통령 표창장, 재외공관의 건물 등에 대한민국의 상징으로 사용하고 있답니다.
----------------------------------------------------------------------------------------------------
document 3:

애국가 1절
동해물과 백두산이 마르고 닳도록
하느님이 보우하사 우리나라 만세
무궁화 삼천리 화려 강산
대한 사람 대한으로 길이 보전하세
----------------------------------------------------------------------------------------------------
document 4:

국새
국새는 우리나라의 도장이에요.
옛날에는 어보, 어새, 옥새, 국새 등 다양한 이름으로 불리어졌지만 현대에는 국새로 부른답니다.
국새를 찍는다는 것은 나라에서 중요한 결정을 한다는 의미로 헌법 개정 공포문의 전문, 외교문서, 훈장증 등에 사용하고 있어요.
현재 사용하고 있는 제5대 국새는 가로, 세로 10.4cm 정사각형으로 무게는 3.38kg이에요. 손잡이는 두 마리

## Creating a Pipeline (Compressor + Document Converter)

`DocumentCompressorPipeline` 을 사용하면 여러 compressor를 순차적으로 결합할 수 있습니다.

`BaseDocumentTransformer`를 파이프라인에 추가할 수 있는데, 이는 맥락적 압축을 수행하지 않고 단순히 문서 집합에 대한 변환을 수행합니다.


In [29]:
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.document_transformers import EmbeddingsRedundantFilter
from langchain_text_splitters import CharacterTextSplitter

# 1.문자 기반 텍스트 분할기 생성 하며 청크 크기를 100으로 청간 중복을 0 으로 설정 합니다.
splitter = CharacterTextSplitter(chunk_size=100, chunk_overlap=0)

# 2. 임베딩을 사용하여 중복 필터를 생성
redundant_filter = EmbeddingsRedundantFilter(embeddings=embeddings)

# 3. 임베딩을 사용하여 관련성 필터를 생성하고 유사도 임계값을 0.86 으로 생성
relevant_filter = EmbeddingsFilter(embeddings=embeddings, similarity_threshold=0.86)

# 4. 문서 압축 파이프라인을 생성하고 분할기, 중복 필터, 관련성 필터 , LLMChain 을 변환기로 설정 
pipeline_compressor = DocumentCompressorPipeline(
    transformers=[
        splitter,
        redundant_filter,
        relevant_filter,
        LLMChainExtractor.from_llm(llm),
    ]
)

While initializing the  `ContextualCompressionRetriever`, we use `pipeline_compressor` as the `base_compressor` and `retriever` as the `base_retriever`.

In [32]:
# 5. 기본 압축기로 pipeline_compressor 설정하고 기본 검색기로 retriever 설정하여 ContextualCompressionRetriever 초기화
compression_retriever = ContextualCompressionRetriever(
    base_compressor=pipeline_compressor,
    base_retriever=retriever,
)

# 6. 질의
compressed_docs = compression_retriever.invoke(
    "우리나라의 대해 알려줘"
)

# 7. Print the relevant documents
pretty_print_docs(compressed_docs)


'''
[ 출력 결과 ]
document 1:

우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요. 영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.

'''

document 1:

우리나라의 정식 이름은 “대한민국”이에요. 사용의 편의상 줄여서 부를 때에는 “대한” 또는 “한국”으로 쓸 수 있어요. 영문으로는 “REPUBLIC OF KOREA”로 쓴답니다.
